In [1]:
%load_ext dotenv
%dotenv

import os

%cd {os.getenv("PROJECT_ROOT") or ".."}

%load_ext autoreload
%autoreload 1

from IPython.display import display

/home/aris/projects/evagpt


In [2]:
import logging

import pandas as pd

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=os.cpu_count(), progress_bar=True, verbose=0)

In [4]:
def show_df(df: pd.DataFrame) -> None:
    display(df.head())
    print(df.shape)

In [5]:
from datasets import load_dataset

dataset = load_dataset("Skylion007/openwebtext", num_proc=8)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Skylion007/openwebtext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Skylion007/openwebtext/79d93d786212f7344586290adb811d4ae6a1762c/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Skylion007/openwebtext/resolve/79d93d786212f7344586290adb811d4ae6a1762c/openwebtext.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Skylion007/openwebtext/Skylion007/openwebtext.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/Skylion007/openwebtext/revision/79d93d786212f7344586290adb811d4ae6a1762c "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Skylion007/openwebtext/resolve/79d93d786212f7344586290adb811d4ae6a1762c/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:htt

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Skylion007/openwebtext/resolve/79d93d786212f7344586290adb811d4ae6a1762c/dataset_infos.json "HTTP/1.1 404 Not Found"


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

In [6]:
split_dataset = dataset["train"].train_test_split(test_size=0.0005, seed=2357, shuffle=True)

In [7]:
split_dataset["val"] = split_dataset.pop("test")  # rename the test split to val

In [8]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [9]:
def process(example):
    ids = tokenizer.encode_ordinary(example["text"])
    ids.append(tokenizer.eot_token)
    out = {"ids": ids, "len": len(ids)}
    return out


tokenized = split_dataset.map(
    process,
    remove_columns=["text"],
    num_proc=32,
)

In [10]:
tokenized

DatasetDict({
    train: Dataset({
        features: ['ids', 'len'],
        num_rows: 8009762
    })
    val: Dataset({
        features: ['ids', 'len'],
        num_rows: 4007
    })
})

In [13]:
import numpy as np

np.sum(tokenized["train"]["len"])

np.int64(9035582489)

In [18]:
from tqdm.notebook import tqdm

for split, dset in tokenized.items():
    arr_len = np.sum(dset["len"], dtype=np.uint64)
    filename = os.path.join("./data/openwebtext", f"{split}.bin")
    dtype = np.uint16  # (can do since enc.max_token_value == 50256 is < 2**16)
    arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))
    total_batches = 1024

    idx = 0
    for batch_idx in tqdm(range(total_batches), desc=f"writing {filename}"):
        # Batch together samples for faster write
        batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format("numpy")
        arr_batch = np.concatenate(batch["ids"])
        # Write into mmap
        arr[idx : idx + len(arr_batch)] = arr_batch
        idx += len(arr_batch)
    arr.flush()

writing ./data/openwebtext/train.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

writing ./data/openwebtext/val.bin:   0%|          | 0/1024 [00:00<?, ?it/s]